# 6.3.4 Hands-on Multilingual Models

Let's try out building a multilingual model.

## 6.3.4.1 Downloading and installing

Like previously, we instal all necessary code

In [1]:
!wget https://www.ccl.kuleuven.be/~vincent/MTAT/plot_train_val_new.py
!wget http://ccl.kuleuven.be/~vincent/MTAT/transformer.py
from IPython.display import Image, display
!pip -q install -U evaluate sacrebleu
!pip install sentencepiece
!wget https://www.ccl.kuleuven.be/~vincent/MTAT/sentencepiece_train.py

--2026-03-03 13:04:09--  https://www.ccl.kuleuven.be/~vincent/MTAT/plot_train_val_new.py
Resolving www.ccl.kuleuven.be (www.ccl.kuleuven.be)... 134.58.16.193
Connecting to www.ccl.kuleuven.be (www.ccl.kuleuven.be)|134.58.16.193|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 8180 (8.0K)
Saving to: ‘plot_train_val_new.py’

plot_train_val_new. 100%[===================>]   7.99K  --.-KB/s    in 0s      

2026-03-03 13:04:10 (131 MB/s) - ‘plot_train_val_new.py’ saved [8180/8180]

--2026-03-03 13:04:10--  http://ccl.kuleuven.be/~vincent/MTAT/transformer.py
Resolving ccl.kuleuven.be (ccl.kuleuven.be)... 134.58.16.193
Connecting to ccl.kuleuven.be (ccl.kuleuven.be)|134.58.16.193|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 16755 (16K)
Saving to: ‘transformer.py’

transformer.py      100%[===================>]  16.36K  --.-KB/s    in 0.1s    

2026-03-03 13:04:11 (139 KB/s) - ‘transformer.py’ saved [16755/16755]

   ━━━━━━━━━━━━━━━━━━━━━━━━

## 6.3.4.2 Add target language tags to the data

We will now train the model on the English-Dutch data, in two directions, and on the French-Dutch
data, also in two directions.
So, we need to tell the model which is the target language, so it can learn to translate in a specific
language

We therefore define the python function `add_targetlang_prefix` that takes three input arguments:
1. the name of the source file
2. the language tag to add
3. the name of the output file

In [2]:
def add_targetlang_prefix (file,target,outputfile):
    f=open(file)
    lines=f.readlines()
    lines_tgt=[f"{target} {s}" for s in lines]
    with open(outputfile, "w") as f:
        for line in lines_tgt:
            f.write(line)

With this function, we create variants of the training and development files with the target
language tag. As shown in the table below, we do this for the train and dev files in both language
directions, for English-Dutch as well as for French-Dutch.

|Original File|Tag|Ouput File |
|-------------|---|-----------|
|tatoeba-en-nl/train.en|<2nl>| en-nl-train.en_2nl|
|tatoeba-en-nl/dev.en|<2nl>| en-nl-dev.en_2nl|
|tatoeba-fr-nl/train.fr|<2nl>| fr-nl-train.fr_2nl|
|tatoeba-fr-nl/dev.fr|<2nl> |fr-nl-dev.fr_2nl|
|tatoeba-en-nl/train.nl|<2en>| en-nl-train.nl_2en|
|tatoeba-en-nl/dev.nl|<2en> |en-nl-dev.nl_2en|
|tatoeba-fr-nl/train.nl|<2fr>| fr-nl-train.nl_2fr|
|tatoeba-fr-nl/dev.nl|<2fr> |fr-nl.dev.nl_2fr|

In [4]:
add_targetlang_prefix("/kaggle/input/tatoeba-en-nl/train.en","<2nl>","en-nl-train.en_2nl")
add_targetlang_prefix("/kaggle/input/tatoeba-en-nl/dev.en","<2nl>","en-nl-dev.en_2nl")
add_targetlang_prefix("/kaggle/input/tatoeba-fr-nl-prepared/tatoeba-fr-nl/train.fr","<2nl>","fr-nl-train.fr_2nl")
add_targetlang_prefix("/kaggle/input/tatoeba-fr-nl-prepared/tatoeba-fr-nl/dev.fr","<2nl>","fr-nl-dev.fr_2nl")

## REVERSE DIRECTIONS
add_targetlang_prefix("/kaggle/input/tatoeba-en-nl/train.nl","<2en>","en-nl-train.nl_2en")
add_targetlang_prefix("/kaggle/input/tatoeba-en-nl/dev.nl","<2en>","en-nl-dev.nl_2en")
add_targetlang_prefix("/kaggle/input/tatoeba-fr-nl-prepared/tatoeba-fr-nl/train.nl","<2fr>","fr-nl-train.nl_2fr")
add_targetlang_prefix("/kaggle/input/tatoeba-fr-nl-prepared/tatoeba-fr-nl/dev.nl","<2fr>","fr-nl-dev.nl_2fr")



## 6.3.4.3 Concatenate all training files and all dev files

Now we need to create two multilingual parallel corpora, which serve as the source and target
side of the training and development data respectively. Note that in the source side each sentence
is prefixed with a target language tag.

We do this by carefully concatenating the files so that the target sentences accord with the target
language tag of the aligned source sentence.

We can use the linux cat command which takes any number of files as arguments and writes
them to the output file specified after the > symbol (redirecting the standard output)

In [5]:
!cat en-nl-train.en_2nl fr-nl-train.fr_2nl en-nl-train.nl_2en fr-nl-train.nl_2fr > train.2multi.src
!cat /kaggle/input/tatoeba-en-nl/train.nl \
    /kaggle/input/tatoeba-fr-nl-prepared/tatoeba-fr-nl/train.nl \
    /kaggle/input/tatoeba-en-nl/train.en \
    /kaggle/input/tatoeba-fr-nl-prepared/tatoeba-fr-nl/train.fr > train.2multi.tgt

!cat en-nl-dev.en_2nl fr-nl-dev.fr_2nl en-nl-dev.nl_2en fr-nl-dev.nl_2fr > dev.2multi.src
!cat /kaggle/input/tatoeba-en-nl/dev.nl\
    /kaggle/input/tatoeba-fr-nl-prepared/tatoeba-fr-nl/dev.nl\
    /kaggle/input/tatoeba-en-nl/dev.en \
    /kaggle/input/tatoeba-fr-nl-prepared/tatoeba-fr-nl/dev.fr > dev.2multi.tgt


## 6.3.4.4 Train a joined SentencePiece Model

In the next step, we train a joined SentencePiece model for all the languages. We first concatenate the source and the target train files into one big file and train SentencePiece on that file.

(Concatenating source and target is maybe unnecessary, we could experimentally test whether this actually helps or not).

In [6]:
!cat train.2multi.src train.2multi.tgt > train.2multi.srctgt
!python sentencepiece_train.py train.2multi.srctgt spm.multi

sentencepiece_trainer.cc(78) LOG(INFO) Starts training with : 
trainer_spec {
  input: train.2multi.srctgt
  input_format: 
  model_prefix: spm.multi
  model_type: UNIGRAM
  vocab_size: 8000
  self_test_sample_size: 0
  character_coverage: 0.9995
  input_sentence_size: 0
  shuffle_input_sentence: 1
  seed_sentencepiece_size: 1000000
  shrinking_factor: 0.75
  max_sentence_length: 4192
  num_threads: 16
  num_sub_iterations: 2
  max_sentencepiece_length: 16
  split_by_unicode_script: 1
  split_by_number: 1
  split_by_whitespace: 1
  split_digits: 0
  pretokenization_delimiter: 
  treat_whitespace_as_suffix: 0
  allow_whitespace_only_pieces: 0
  required_chars: 
  byte_fallback: 0
  vocabulary_output_piece_score: 1
  train_extremely_large_corpus: 1
  seed_sentencepieces_file: 
  hard_vocab_limit: 1
  use_all_vocab: 0
  unk_id: 0
  bos_id: 1
  eos_id: 2
  pad_id: -1
  unk_piece: <unk>
  bos_piece: <s>
  eos_piece: </s>
  pad_piece: <pad>
  unk_surface:  ⁇ 
  enable_differential_privacy: 0

## 6.3.4.5 Actual training the MNMT system

Now we can run the actual training. Note that we can use the same script as before, so only the data is changed compared to the single translation-pair approach, the training algorithm stays exactly as it was.

In [ ]:
!python transformer.py \
  --src-file train.2multi.src \
  --tgt-file train.2multi.tgt \
  --src-val  dev.2multi.src \
  --tgt-val  dev.2multi.tgt \
  --spm-src-model spm.multi.model \
  --spm-tgt-model spm.multi.model \
  --save multi_nmt \
  --epochs 10 \
  --eval-metrics \
  --show-val-examples 10 \
  --history-json transformer.multi.hist \
  --eval-metrics


E0000 00:00:1772543098.217262     416 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772543098.268490     416 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772543098.675337     416 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772543098.675381     416 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772543098.675385     416 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772543098.675389     416 computation_placer.cc:177] computation placer already registered. Please check linka

## 6.3.4.6 Compare results with previous approaches

In [ ]:
!python plot_train_val_new.py transformer.multi.hist \
--train-key loss \
--val-key eval_loss \
--save tfm_train_val.png

display(Image(filename="tfm_train_val.png"))

In [ ]:
!python plot_train_val_new.py transformer.multi.hist \
  --val-key eval_bleu \
  --save tfm_bleu.png

display(Image(filename="tfm_bleu.png"))

We can see that the BLEU scores are now converging just below 50, whereas previously, we converged just a bit higher. While this seems to indicate that we lost some of the translation quality compared to single translation-pair approach, we should be aware that results are not fully comparable, as here results are presented for the combined development set that contains four different translation directions, whereas results in the previous session only evaluated a single translation direction.

## 6.3.4.7 Testing the zero-shot Language Pair

Now we have build a (small) multilingual model. So that was Phase 1. Now we
enter Phase 2, in which we actually test the English-to-French translation, for which no examples
were seen in the training data.

Up till now, we have not yet translated with trained models outside of the training script. We
can use the transfomer_translate.py script to actually translate files.

When we want to translate dev.en into French, we need to prefix the sentences with the `<2fr>` tag.
We can then run the `transformer_translate.py` script that takes the arguments shown in the table below.


|Argument|Default|Description|
|--------|-------|-----------|
|`–model-dir`| - | Directory passed as `–save` in `transformer.py`|
|`–src-file` | - | Source file to translate|
|`–out-file` | - | Output translations file|
|`–batch-size`| 32 | Batch size during translation|
|`–max-src-len`| 128 | Maximum nr of tokens in source|
|`–max-gen-len`| 128 | Maximum length of translation |
|`–num-beams`  |  4  |Number of alternatives under consideration|

In [ ]:
!wget https://raw.githubusercontent.com/VincentCCL/MTAT/refs/heads/main/code/transformer_translate.py
add_targetlang_prefix("/kaggle/input/tatoeba-en-nl/dev.en","<2fr>","en-nl-dev.en_2fr")
!python transformer_translate.py --model multi_nmt --src-file en-nl-dev.en_2fr --out-file en-nl-dev.en_2fr.zeroshot.fr 

After running the script, results can be found in `en-nl-dev.en_2fr.zeroshot.fr`.

In order to manually check the translation quality, we can peak inside the source file and the
resulting translation file, using the `head` command

In [ ]:
!head -n 20 en-nl-dev.en_2fr

In [ ]:
!head -n 20 en-nl-dev.en_2fr.zeroshot.fr

We can see that the translation coming out of this model are not very good when translation from English to French, but we can also see that the model has learned a little bit to do this.

The main reasons why the resulting model is not very good is due to the limited scale of the Tatoeba dataset, so there is not enough opportunity to learn proper representations of the languages in the zero-shot case.
